# chatbot evaluation


In [75]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"


In [76]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbots3 Evaluation"

dataset = client.create_dataset(dataset_name)

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {
                "question": "What is LangChain?"
            },
            "outputs": {
                "answer": "A framework for building LLM applications"
            },
        },
        {
            "inputs": {
                "question": "What is LangSmith?"
            },
            "outputs": {
                "answer": "A platform for observing and evaluating LLM applications"
            },
        },
        {
            "inputs": {
                "question": "What is OpenAI?"
            },
            "outputs": {
                "answer": "A company that creates Large Language Models"
            },
        },
        {
            "inputs": {
                "question": "What is Google?"
            },
            "outputs": {
                "answer": "A technology company known for search"
            },
        },
        {
            "inputs": {
                "question": "What is Mistral?"
            },
            "outputs": {
                "answer": "A company that creates Large Language Models"
            },
        }
    ]
)

{'example_ids': ['33a8602b-f91d-47e3-bc37-eb92cfe763a0',
  '837b1097-b42e-4d28-809d-d8cddfdfa361',
  '310c4512-5563-4d7a-8b89-d2772d9f3bce',
  '605eee20-fada-4fbf-97a2-48368bf7e9da',
  'ca224800-6fb4-4047-b761-4d94d378032a'],
 'count': 5,
 'as_of': '2026-09-09T15:51:05.093118789Z'}

### defining metrics(llm as a judge)

In [ ]:

from groq import Groq

groq_client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)


# --------------------------------------------------
# Evaluation instructions
# --------------------------------------------------

eval_instructions = """
You are an expert professor specialized in grading
students' answers to questions.

Compare the predicted answer with the real answer.

If the predicted answer is semantically correct,
respond with exactly:

CORRECT

Otherwise respond with exactly:

INCORRECT
"""


# --------------------------------------------------
# Correctness evaluator
# --------------------------------------------------

def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict
) -> bool:

    user_content = f"""
Question:
{inputs["question"]}

Real answer:
{reference_outputs["answer"]}

Predicted answer:
{outputs["response"]}

Is the predicted answer semantically correct?

Respond with only:
CORRECT
or
INCORRECT
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": eval_instructions
            },
            {
                "role": "user",
                "content": user_content
            }
        ]
    )

    result = response.choices[0].message.content

    return result.strip().upper() == "CORRECT"


In [78]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(
    outputs: dict,
    reference_outputs: dict
) -> bool:

    return len(outputs["response"]) < (
        2 * len(reference_outputs["answer"])
    )


### run evaluation

In [79]:
default_instructions = """
Respond to the user's question in a short,
concise manner (one short sentence).
"""


def my_app(
    question: str,
    model: str = "openai/gpt-oss-20b",
    instructions: str = default_instructions
) -> str:

    response = groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": instructions
            },
            {
                "role": "user",
                "content": question
            }
        ]
    )

    return response.choices[0].message.content

In [80]:
### Call my_app for every datapoints
def ls_target(inputs: dict) -> dict:

    return {
        "response": my_app(inputs["question"])
    }


In [81]:
## Run our evaluation
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[
        correctness,
        concision
    ],
    experiment_prefix="groq-gpt-oss-20b-chatbot"
)

print(experiment_results)

View the evaluation results for experiment: 'groq-gpt-oss-20b-chatbot-0f5e25cb' at:
https://smith.langchain.com/o/dad397bb-fd64-4cb1-b11e-2d25100bd00c/datasets/156950ec-2a67-4eef-8436-a56e28b8e164/compare?selectedSessions=fe5e97a2-b174-4852-85da-4caeee45d081




0it [00:00, ?it/s]

<ExperimentResults groq-gpt-oss-20b-chatbot-0f5e25cb>


## RAG Eval

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

#list of URLS to load documents from 
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]
